# Session 2, Block 3 &mdash; Statistics & Hypothesis Testing
**Data Science Techniques and Real-World Applications &mdash; WS 2026**

Thursday 10 September 2026 &middot; Block 3 (14:00&ndash;15:30)

Regression itself waits until Day 3 &mdash; today is about the reasoning underneath it: how do you
decide, from a sample of data, whether a pattern you're seeing is real or just noise?


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_excel("data/techstocks.xlsx", sheet_name="main")
df.head()

## <span style="color:#1e3a8a">The logic of hypothesis testing</span>

A **hypothesis test** is a way of using sample data to make a decision about something you can't
observe directly &mdash; the true, underlying pattern in the whole population.

**Purpose:**
- Gather evidence from a dataset.
- Decide whether a claim about the population is plausible.
- Help distinguish a genuine effect from something that could easily be due to chance.

**Key idea:** compare the evidence in your data against a baseline assumption &mdash; you don't
prove your hypothesis directly; you ask "if the boring, no-effect explanation were true, how
surprising would this data be?"


## <span style="color:#1e3a8a">The steps</span>

**1. State the hypotheses**
- **Null hypothesis ($H_0$)**: no effect, no difference. The boring, default explanation.
- **Alternative hypothesis ($H_1$ or $H_A$)**: there is an effect or difference.

**2. Choose a significance level ($\alpha$)** &mdash; commonly 0.05 (a 5% tolerance for error).

**3. Select a test and calculate a test statistic** &mdash; t-test, chi-square, ANOVA, etc.,
depending on the question and the data.

**4. Apply the decision rule**
- If the *p*-value $< \alpha$: reject $H_0$ &mdash; the data would be quite surprising if $H_0$
  were true, so treat this as evidence *for* $H_1$.
- If the *p*-value $\geq \alpha$: do not reject $H_0$ &mdash; not enough evidence against it.
  (This is *not* the same as proving $H_0$ true &mdash; "not enough evidence" and "definitely no
  effect" are different claims.)


## <span style="color:#1e3a8a">Outcomes and errors</span>

Whichever way a test comes out, it can be wrong in one of two distinct ways:

| | $H_0$ is actually true | $H_0$ is actually false |
|---|---|---|
| **Reject $H_0$** | Type I error (false positive), probability = $\alpha$ | Correct |
| **Fail to reject $H_0$** | Correct | Type II error (false negative), probability = $\beta$ |

- **Type I error**: concluding there's an effect when there really isn't (e.g. "this drug works" when it doesn't).
- **Type II error**: missing a real effect (e.g. "this drug doesn't work" when it actually does).
- $1 - \beta$ is called the **power** of the test &mdash; its ability to detect a real effect when one exists.

There's an inherent tension here: making a test more conservative (harder to reject $H_0$) lowers
your Type I error rate but raises your Type II error rate, and vice versa. Neither error can be
driven to zero at the same time.


## <span style="color:#1e3a8a">The t-test</span>

The most common test for "is this average meaningfully different from some value (often zero), or
different between two groups?"

- Start by assuming $H_0$ is true: the parameter of interest equals 0.
- Collect evidence: is the estimated value ($\hat s$) close to 0, or far from it?
- The **t-statistic** measures "how far," in standard-error units:

$$t = \frac{\bar x}{SE(\bar x)} \qquad \text{(one-sample: is the mean 0?)}$$

$$t = \frac{\bar x_A - \bar x_B}{SE(\bar x_A - \bar x_B)} \qquad \text{(two-sample: are the means equal?)}$$

`scipy.stats` computes both the t-statistic and the p-value for you &mdash; you rarely need to
compute either by hand.


### One-sample t-test: is AAPL's average daily return actually different from zero?

In [ ]:
aapl_returns = df[df["Ticker Symbol"] == "AAPL"]["Returns"]

t_stat, p_value = stats.ttest_1samp(aapl_returns, popmean=0)
print(f"mean return: {aapl_returns.mean():.5f}")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value:     {p_value:.4f}")

If the p-value is below 0.05, that's evidence AAPL's average daily return over this period was
not zero -- notice this says nothing about *why*, or whether it will continue; it's a statement about
this sample, evaluated against a specific null hypothesis.

### Two-sample t-test: is AAPL's average return different from GOOG's?

In [ ]:
aapl = df[df["Ticker Symbol"] == "AAPL"]["Returns"]
goog = df[df["Ticker Symbol"] == "GOOG"]["Returns"]

t_stat, p_value = stats.ttest_ind(aapl, goog)
print(f"AAPL mean: {aapl.mean():.5f}   GOOG mean: {goog.mean():.5f}")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value:     {p_value:.4f}")

### Confidence intervals

A p-value answers "is this significant?" A **confidence interval** answers the arguably more useful
question: "what's a plausible *range* for the true value?" A 95% CI is the range of values that a
hypothesis test at $\alpha = 0.05$ would *not* reject.


In [ ]:
ci = stats.t.interval(
    confidence=0.95,
    df=len(aapl_returns) - 1,
    loc=aapl_returns.mean(),
    scale=stats.sem(aapl_returns),   # standard error of the mean
)
print(f"95% CI for AAPL's mean daily return: ({ci[0]:.5f}, {ci[1]:.5f})")

<span style="color:#b45309">**Exercise 1: Test another pair of tickers**</span>

Run a two-sample t-test comparing `Returns` for AMZN and NFLX. State the null hypothesis in words, report the p-value, and say whether you'd reject $H_0$ at $\alpha = 0.05$.

Try it in the cell below, then check the answer notebook (`Session 2c - Exercise Answers.ipynb`).


In [ ]:
# your code here


## <span style="color:#1e3a8a">A closing caution: testing many hypotheses at once</span>

If you run 20 independent hypothesis tests at $\alpha = 0.05$, you'd expect **about one of them to
come back "significant" by pure chance**, even if nothing real is going on anywhere. This is exactly
why it pays to be conservative about which hypotheses you test, and skeptical of a single significant
result buried among many tests that weren't reported.

This isn't a technicality &mdash; it's one of the most common ways a well-intentioned analysis
misleads. Keep it in mind for every case study this term: report what you tested, not just what
came out significant.
